# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kubrafathima2005-kf/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content page belonging to one client on one reporting date.

**Table:** `fact_content_daily_performance`.

**Time window:** March 2026 (`month=2026-03`), a mid-panel month used for development and verification.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:** Past-period search and engagement performance values that are available before prediction.

**Label / proxy:** Future content performance decline, measured using a clearly defined future time window.

**Context:** `report_date`, `client_id`, and `content_id`. These identify the date, client, and content item; they are not model features.

**Excluded:** Future-period metrics and label-derived fields, because they would cause data leakage.

**Context:** `report_date`, `client_hash_id`, and `content_hash_id`. These identify the reporting date, client, and content item. They are not model features.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:

%pip -q install duckdb

import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

table = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/'
    'fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("Warehouse connection setup complete!")


Warehouse connection setup complete!


In [27]:
from huggingface_hub import list_repo_files

all_files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

feb_files = [
    f for f in all_files
    if f.startswith("fact_content_daily_performance/month=2026-02/")
    and f.endswith(".parquet")
]

print("February files found:", len(feb_files))
print(feb_files[:5])

February files found: 1
['fact_content_daily_performance/month=2026-02/data_0.parquet']


**Features:** February 2026 historical performance values: `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, and `ga4_sessions`.

**Label / proxy:** March 2026 content performance decline. Label = 1 when March GSC clicks are lower than February GSC clicks; otherwise label = 0.

**Context:** `report_date`, `client_hash_id`, and `content_hash_id`. These identify the reporting date, client, and content item. They are not model features.

**Excluded:** March clicks from the model features because they are used to create the future outcome label and would cause data leakage.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [30]:

print("Token loaded:", bool(hf_token))
print("DuckDB version:", con.sql("SELECT version()").fetchone()[0])


Token loaded: True
DuckDB version: v1.3.2


In [31]:

con.execute("DROP SECRET IF EXISTS hf")

con.execute(f"""
CREATE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

print("Hugging Face authentication configured!")


Hugging Face authentication configured!


In [32]:

hf_token = userdata.get("HF_TOKEN").strip()
print("Token loaded:", bool(hf_token))


Token loaded: True


In [33]:

from huggingface_hub import HfApi

api = HfApi(token=hf_token)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

march_files = [
    f for f in files
    if "fact_content_daily_performance/month=2026-03/" in f
]

print("March files found:", len(march_files))
print(march_files[:10])


March files found: 1
['fact_content_daily_performance/month=2026-03/data_0.parquet']


In [34]:

from huggingface_hub import hf_hub_url

file_url = hf_hub_url(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

con.execute("""
CREATE OR REPLACE SECRET hf_http (
    TYPE http,
    EXTRA_HTTP_HEADERS MAP {
        'Authorization': 'Bearer """ + hf_token + """'
    }
)
""")

schema = con.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{file_url}')"
).df()

display(schema)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [37]:

print(schema["column_name"].tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [38]:

# QUERY 1: Check duplicate rows
q1 = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{file_url}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

print("Query 1: Duplicate check")
display(q1)


# QUERY 2: Check total rows and date range
q2 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{file_url}')
""").df()

print("Query 2: Row count and date range")
display(q2)


# QUERY 3: Check GSC data availability
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows
FROM read_parquet('{file_url}')
""").df()

print("Query 3: GSC availability")
display(q3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1: Duplicate check


,report_date,client_hash_id,content_hash_id,row_count


Query 2: Row count and date range


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3: GSC availability


,total_rows,available_rows
0,9841378,3611061


In [39]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

data = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(ga4_sessions) AS ga4_sessions
    FROM read_parquet('{feb_file_url}')
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS mar_clicks
    FROM read_parquet('{file_url}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    feb.*,
    mar.mar_clicks,
    CASE WHEN mar.mar_clicks < feb.gsc_clicks
         THEN 1 ELSE 0 END AS decline_label
FROM feb
INNER JOIN mar USING (client_hash_id, content_hash_id)
""").df()

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

X = data[features]
y = data["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, model.predict(X_test))

X_leaky = data[features + ["mar_clicks"]]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42, stratify=y
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

leaky_model.fit(X_train_l, y_train_l)
leaky_score = accuracy_score(y_test_l, leaky_model.predict(X_test_l))

print("Dataset rows:", len(data))
print("Honest model accuracy:", round(honest_score, 4))
print("Leaky model accuracy:", round(leaky_score, 4))
print("Leakage feature: March clicks (future information)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset rows: 21863
Honest model accuracy: 0.8413
Leaky model accuracy: 0.9339
Leakage feature: March clicks (future information)


Only 21,863 matched records had both GSC and GA4 data available. The results may not represent all content items."

### Five candidate features

1. `gsc_impressions` — Known from Google Search Console data recorded before the prediction.
2. `gsc_clicks` — Known from historical search clicks.
3. `gsc_avg_position` — Known from historical average search ranking.
4. `ga4_pageviews` — Known from historical pageview data.
5. `ga4_sessions` — Known from historical website session data.

These features must be taken from an earlier period than the prediction date.

In [40]:

import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Create a small demonstration dataset
np.random.seed(42)

X = np.random.rand(500, 3)
y = (X[:, 0] + X[:, 1] > 1).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Honest model
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, model.predict(X_test))

# Deliberate leakage: give the model the answer itself
X_leaky = np.column_stack([X, y])

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42
)

leaky_model = DecisionTreeClassifier(random_state=42)
leaky_model.fit(X_train_l, y_train_l)
leaky_score = accuracy_score(
    y_test_l, leaky_model.predict(X_test_l)
)

print("Honest accuracy:", honest_score)
print("Leaked accuracy:", leaky_score)
print("Leakage makes the score misleadingly perfect.")


Honest accuracy: 0.97
Leaked accuracy: 1.0
Leakage makes the score misleadingly perfect.


### Self-check

- [x] Defined the data contract.
- [x] Identified features, label, context and excluded fields.
- [x] Checked the data grain and duplicate rows.
- [x] Verified the March 2026 date range and row count.
- [x] Checked GSC data availability.
- [x] Listed five candidate features.
- [x] Demonstrated data leakage.
- [x] Documented one limitation of the data slice.

### Limitation of the March 2026 slice

March 2026 is only one month's data. It may not represent seasonal changes or performance patterns in other months. Therefore, results from this slice may not generalize to all future periods.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.